# StepManAI — Training

**Just press the single ▶ button below and wait.** (Allow Google Drive access when asked.)

- **First ever run:** downloads the 19 official DDR packs, builds the dataset (~1-1.5 h), saves it to your Drive, then trains (~1 h). This only happens once.
- **Later runs:** load the dataset from Drive in ~2 min and go straight to training.
- **When it says ALL DONE:** the models are in `Drive/StepManAI/checkpoints/` — open the generate notebook and make charts. You never need this notebook again.

In [ ]:
#@title Train StepManAI (press ▶ and wait)
placement_epochs = 40 #@param {type:"integer"}
selection_epochs = 30 #@param {type:"integer"}

import glob, os, subprocess, sys, torch
assert torch.cuda.is_available(), 'No GPU! Menu: Runtime -> Change runtime type -> T4 GPU, then re-run.'
print('GPU:', torch.cuda.get_device_name(0))

# --- code + Drive ---
if not os.path.exists('/content/StepManAI'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/Mrman67/StepManAI.git', '/content/StepManAI'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'soundfile'], check=True)
from google.colab import drive
drive.mount('/content/drive')

# --- dataset: Drive copy if it exists, else build from ZIV once ---
DRIVE_TAR = '/content/drive/MyDrive/StepManAI/stepmanai_dataset.tar'
if os.path.exists('/content/data/labels.pkl'):
    print('dataset already prepared in this session')
elif os.path.exists(DRIVE_TAR):
    print('loading dataset from Drive (~2 min)...')
    !mkdir -p /content/data && tar -xf "{DRIVE_TAR}" -C /content/data
else:
    print('FIRST RUN: downloading 19 DDR packs from Zenius-I-Vanisher...')
    PACKS = {37:'1st', 32:'2nd', 38:'3rd', 39:'4th', 303:'4thplus', 30:'5th',
             40:'max', 31:'max2', 41:'extreme', 1:'supernova', 77:'supernova2',
             295:'x', 546:'x2', 802:'x3', 1148:'a', 1292:'a20', 1293:'a20plus',
             1509:'a3', 1709:'world'}
    os.makedirs('/content/Songs', exist_ok=True)
    for i, (cid, name) in enumerate(PACKS.items()):
        dst = f'/content/Songs/{name}'
        if os.path.exists(dst):
            continue
        print(f'  pack {i+1}/{len(PACKS)}: {name}', flush=True)
        url = f'https://zenius-i-vanisher.com/v5.2/download.php?type=ddrpack&categoryid={cid}'
        !wget -q -O /content/pack.zip "{url}" && mkdir -p "{dst}" && unzip -qo /content/pack.zip -d "{dst}" && rm -f /content/pack.zip
    print('extracting audio features (~30-40 min)...')
    os.environ['STEPMANAI_SONGS'] = '/content/Songs'
    os.environ['STEPMANAI_INDEX'] = '/content/data/index.json'
    os.environ['STEPMANAI_CACHE'] = '/content/data/cache_u8'
    os.environ['STEPMANAI_LABELS'] = '/content/data/labels.pkl'
    os.environ['STEPMANAI_U8'] = '1'
    !cd /content/StepManAI && python scan_library.py && python build_cache.py
    print('saving dataset to Drive so this never runs again...')
    !mkdir -p /content/drive/MyDrive/StepManAI
    !tar -cf "{DRIVE_TAR}" -C /content/data cache_u8 labels.pkl
print(len(glob.glob('/content/data/cache_u8/*.npy')), 'songs ready')

# --- train both models ---
os.environ['STEPMANAI_ROOT'] = '/content/StepManAI'
os.environ['STEPMANAI_CACHE'] = '/content/data/cache_u8'
os.environ['STEPMANAI_LABELS'] = '/content/data/labels.pkl'
os.environ['BATCH'] = '64'
os.environ['EPOCHS'] = str(placement_epochs)
print('\n=== training placement model (when steps happen) ===')
!cd /content/StepManAI && python train_placement.py
os.environ['EPOCHS'] = str(selection_epochs)
print('\n=== training selection model (which arrows) ===')
!cd /content/StepManAI && python train_selection.py

# --- save to Drive ---
!mkdir -p /content/drive/MyDrive/StepManAI/checkpoints
!cp /content/StepManAI/checkpoints/placement.pt /content/StepManAI/checkpoints/selection.pt /content/drive/MyDrive/StepManAI/checkpoints/
print('\nALL DONE — models saved to Drive/StepManAI/checkpoints/. Open the generate notebook!')